# Prepare physically oriented CMEP volumes

This notebook prepares independent plan-view and cross-sectional intensity volumes, supports non-destructive manual calibration, optimizes their correlative overlap, constructs a correlative likelihood score map, and localizes subvoxel atomic centers. It stops before atom classification or volume fusion. Raw TIFF files are opened read-only.

In [ ]:
from pathlib import Path

import importlib
import numpy as np

import cmep_alignment
import cmep_alignment_viewer
import cmep_likelihood
import cmep_atom_localization
import cmep_gpu_viewer
import cmep_atom_viewer
import cmep_registration
import cmep_volume
import cmep_visualization

importlib.reload(cmep_alignment)
importlib.reload(cmep_alignment_viewer)
importlib.reload(cmep_likelihood)
importlib.reload(cmep_atom_localization)
importlib.reload(cmep_gpu_viewer)
importlib.reload(cmep_atom_viewer)
importlib.reload(cmep_registration)
importlib.reload(cmep_volume)
importlib.reload(cmep_visualization)

from cmep_alignment_viewer import launch_alignment_viewer
from cmep_atom_localization import localize_atoms, print_atom_localization_summary
from cmep_atom_viewer import (
    write_cpu_atom_marker_viewer,
    write_gpu_atom_marker_viewer,
    write_gpu_atom_viewer,
)
from cmep_gpu_viewer import write_gpu_volume_viewer
from cmep_likelihood import (
    create_likelihood_map,
    make_likelihood_histogram,
    print_likelihood_summary,
)
from cmep_registration import optimize_correlative_alignment, print_registration_summary
from cmep_volume import process_volume, print_volume_summary
from cmep_visualization import visualize_volume

## Configuration

The TIFF tags do not contain a physical calibration. Confirm the user-editable pixel sizes, stack depths, unit, and lattice parameter below before scientific interpretation. `depth_plan` and `depth_cross` are the distances between the centers of the first and last original slices.

In [ ]:
project_dir = Path.cwd()

# Each input may instead be an already loaded 3D NumPy array.
plan_input = project_dir / 'Plan-view_slices_0.39.tif'
cross_input = project_dir / 'Cross-section_slices_0.39.tif'

ori_plan = 'yx'
ori_cross = 'zx'

length_unit = 'nm'
px_size_plan = 18.74 / 1000
px_size_cross = 18.74 / 1000
lat_param = 0.3905
depth_plan = 25 * lat_param
depth_cross = 29 * lat_param

flip_x_plan = False
flip_y_plan = False
flip_z_plan = False
flip_x_cross = True
flip_y_cross = False
flip_z_cross = False

color_plan = 'lightcoral'
color_cross = 'lightblue'

# Viewer controls. Normalized intensities have the fundamental range [0, 1].
threshold_initial = 0.50
threshold_min = 0.2
threshold_max = 0.8
threshold_count = 1  # Set to 1 to use threshold_initial without a slider.
max_display_voxels = 1e6  # Use None to include every thresholded voxel.
voxel_scale = 1.  # Fraction of each voxel's physical spacing.
minimum_voxel_alpha = 0.30  # Opacity at the active threshold; 0=invisible, 1=opaque.
background_color = 'black'  # Axis lines and text use its RGB inverse.
viewer_size_px = 800  # Equal width and height gives a 1:1 viewer canvas.

# Initial alignment viewer. Its slider spans the union in physical depth.
alignment_initial_plane = 'yx'
alignment_depth_positions = 250
alignment_rotation_min_degrees = -180.0
alignment_rotation_max_degrees = 180.0
alignment_rotation_step_degrees = 0.25
alignment_state_path = project_dir / 'alignment_states' / 'cmep_initial_alignment.json'
alignment_load_saved_state = True

# Correlative registration. All distances use length_unit.
registration_moving_dataset = 'cross'
coarse_search_axes = 'xy'
coarse_search_radius = 2
coarse_translation_step = lat_param / 2 # There may be Sr/Ti confusion
coarse_optimization_spacing = lat_param / 5
refinement_optimization_spacing = lat_param / 15
fine_translation_limit = 0.15
fine_translation_initial_step = 0.05
fine_translation_tolerance = 0.00625
# Fine world-frame rotation-vector settings in physical (x, y, z) order.
# Set an axis limit to 0 to disable fine rotation about that direction.
fine_rotation_limit_degrees_xyz = (1.5, 1.5, 1.5)
fine_rotation_initial_step_degrees_xyz = (0.75, 0.75, 0.75)
fine_rotation_tolerance_degrees_xyz = (0.05, 0.05, 0.05)
registration_intensity_floor = 0.10
registration_chunk_voxels = 500_000
registration_native_validation = True
registration_materialize_aligned_grid = True
optimized_alignment_state_path = project_dir / 'alignment_states' / 'cmep_correlative_alignment.json'
optimized_aligned_grid_path = project_dir / 'alignment_states' / 'cmep_correlative_aligned_grid.npz'

## Process both stacks

Each result follows the same pipeline regardless of whether its input is a TIFF path or a NumPy array. The normalized stacks remain `(slice, row, col)`; the final volumes use canonical `(x, y, z)` order.

In [ ]:
normalized_plan, volume_plan, meta_plan = process_volume(
    plan_input,
    dataset_kind='plan_view',
    orientation=ori_plan,
    pixel_size=px_size_plan,
    physical_depth=depth_plan,
    length_unit=length_unit,
    lat_param=lat_param,
    flips={'x': flip_x_plan, 'y': flip_y_plan, 'z': flip_z_plan},
)

normalized_cross, volume_cross, meta_cross = process_volume(
    cross_input,
    dataset_kind='cross_section',
    orientation=ori_cross,
    pixel_size=px_size_cross,
    physical_depth=depth_cross,
    length_unit=length_unit,
    lat_param=lat_param,
    flips={'x': flip_x_cross, 'y': flip_y_cross, 'z': flip_z_cross},
)

N_plan = normalized_plan.shape[0]
N_cross = normalized_cross.shape[0]

x_plan, y_plan, z_plan = (meta_plan['coordinates'][axis] for axis in 'xyz')
x_cross, y_cross, z_cross = (meta_cross['coordinates'][axis] for axis in 'xyz')

In [ ]:
print_volume_summary('Plan-view', meta_plan)
print()
print_volume_summary('Cross-section', meta_cross)

## Interactive 3D view

Choose one dataset before running the cell. The default `gpu` backend writes a self-contained standalone viewer using one shared cube and GPU instancing, then opens it in the system browser; the notebook stores only its path. Set `viewer_backend='plotly'` for the previous inline viewer. The GPU viewer uses `threshold_initial`; `threshold_count` applies only to Plotly's optional slider. Voxel opacity, physical coordinates, colors, and the user-controlled voxel selection limit are shared by both backends. Set `max_display_voxels=None` to include every voxel satisfying the threshold.

In [ ]:
dataset_to_view = 'cross'  # 'plan' or 'cross'
viewer_backend = 'gpu'  # 'gpu' (standalone, instanced) or 'plotly' (inline).

if dataset_to_view == 'plan':
    volume_to_view, meta_to_view, viewer_color = volume_plan, meta_plan, color_plan
elif dataset_to_view == 'cross':
    volume_to_view, meta_to_view, viewer_color = volume_cross, meta_cross, color_cross
else:
    raise ValueError("dataset_to_view must be 'plan' or 'cross'.")

if viewer_backend == 'gpu':
    viewer_output = write_gpu_volume_viewer(
        volume_to_view,
        meta_to_view,
        output_path=project_dir / 'gpu_viewers' / f'{dataset_to_view}_gpu_viewer.html',
        color=viewer_color,
        threshold=threshold_initial,
        max_voxels=max_display_voxels,
        voxel_scale=voxel_scale,
        minimum_voxel_alpha=minimum_voxel_alpha,
        background_color=background_color,
        open_browser=True,
    )
elif viewer_backend == 'plotly':
    viewer_output = visualize_volume(
        volume_to_view,
        meta_to_view,
        color=viewer_color,
        threshold=threshold_initial,
        threshold_min=threshold_min,
        threshold_max=threshold_max,
        threshold_count=threshold_count,
        max_voxels=max_display_voxels,
        voxel_scale=voxel_scale,
        minimum_voxel_alpha=minimum_voxel_alpha,
        background_color=background_color,
        figure_size=viewer_size_px,
    )
else:
    raise ValueError("viewer_backend must be 'gpu' or 'plotly'.")

viewer_output

## Initial 2D calibration, rotation, and translation

This viewer slices both canonical volumes at one shared physical coordinate. The rotation slider acts on the selected dataset about the active plane normal through its transformed geometric center; release the slider to commit and reslice the whole 3D dataset. Directional calibration scales both selected directions to the requested distance, then rotates and translates cross so its selected endpoints coincide with plan. Rotation, calibration, and in-plane dragging update only the source-to-world affine matrices; `volume_plan` and `volume_cross` remain unchanged. Plane buttons use the same first-letter vertical, second-letter horizontal convention as the input orientations.

In [ ]:
# Stop a previous viewer from this kernel before opening a fresh session.
if 'alignment_viewer' in globals():
    alignment_viewer.close()

alignment_viewer = launch_alignment_viewer(
    volume_plan,
    meta_plan,
    volume_cross,
    meta_cross,
    state_path=alignment_state_path,
    initial_plane=alignment_initial_plane,
    color_plan=color_plan,
    color_cross=color_cross,
    threshold_initial=threshold_initial,
    threshold_min=threshold_min,
    threshold_max=threshold_max,
    threshold_count=11,#threshold_count,
    depth_positions=alignment_depth_positions,
    minimum_voxel_alpha=minimum_voxel_alpha,
    background_color=background_color,
    voxel_scale=voxel_scale,
    rotation_min_degrees=alignment_rotation_min_degrees,
    rotation_max_degrees=alignment_rotation_max_degrees,
    rotation_step_degrees=alignment_rotation_step_degrees,
    load_saved_state=alignment_load_saved_state,
    open_browser=True,
)
alignment_viewer

## Correlative overlap optimization

Save the manual alignment state before running this cell. The fixed-grid objective uses floor-subtracted intensities and multiplies their normalized product by signal-support coverage. Fine registration jointly optimizes physical `(x, y, z)` translation and one world-frame three-component rotation vector about the moving dataset's intensity-weighted center of mass. This is an axis-angle/exponential-coordinate rotation, not an ordered sequence of Euler rotations. The source volumes remain unchanged; the result contains composed affine transforms and two aligned intensity arrays on one physical `(x, y, z)` grid.

In [ ]:
registration_result = optimize_correlative_alignment(
    volume_plan,
    meta_plan,
    volume_cross,
    meta_cross,
    alignment_state=alignment_state_path,
    moving_dataset=registration_moving_dataset,
    coarse_search_axes=coarse_search_axes,
    coarse_search_radius=coarse_search_radius,
    coarse_translation_step=coarse_translation_step,
    coarse_optimization_spacing=coarse_optimization_spacing,
    refinement_optimization_spacing=refinement_optimization_spacing,
    fine_translation_limit=fine_translation_limit,
    fine_translation_initial_step=fine_translation_initial_step,
    fine_translation_tolerance=fine_translation_tolerance,
    fine_rotation_limit_degrees_xyz=fine_rotation_limit_degrees_xyz,
    fine_rotation_initial_step_degrees_xyz=fine_rotation_initial_step_degrees_xyz,
    fine_rotation_tolerance_degrees_xyz=fine_rotation_tolerance_degrees_xyz,
    intensity_floor=registration_intensity_floor,
    chunk_voxels=registration_chunk_voxels,
    native_validation=registration_native_validation,
    materialize_aligned_grid=registration_materialize_aligned_grid,
    result_state_path=optimized_alignment_state_path,
    aligned_grid_path=(
        optimized_aligned_grid_path
        if registration_materialize_aligned_grid
        else None
    ),
    progress=True,
)

optimized_alignment_state = registration_result.optimized_alignment_state
optimized_transforms = registration_result.optimized_transforms
aligned_optimization_data = registration_result.aligned_grid

if aligned_optimization_data is not None:
    x_optimized = aligned_optimization_data['coordinates']['x']
    y_optimized = aligned_optimization_data['coordinates']['y']
    z_optimized = aligned_optimization_data['coordinates']['z']
    volume_plan_optimized = aligned_optimization_data['volume_plan']
    volume_cross_optimized = aligned_optimization_data['volume_cross']
    valid_optimized_overlap = aligned_optimization_data['valid_overlap_mask']

print_registration_summary(registration_result)

## Correlative likelihood score map

The aligned NPZ is loaded directly, so this section can be rerun without repeating registration. Each dataset is independently floor-subtracted and rescaled before their geometric mean is calculated. The complete physical correlative likelihood score map is saved as `float32`; `NaN` marks locations outside the valid overlap. The viewer applies no hidden voxel limit, although a very low threshold can still exceed available browser or GPU memory.

In [ ]:
# Correlative likelihood score map. Named colormaps use Plotly colorscale names.
likelihood_floor_plan = 0.1
likelihood_floor_cross = 0.1
likelihood_chunk_voxels = 10_000_000  # Approximate voxels per memory batch; no sampling.
likelihood_threshold = 0.4
likelihood_colormap = 'magma'
likelihood_max_display_voxels = None  # No cap or subsampling.
likelihood_map_path = project_dir / 'alignment_states' / 'cmep_correlative_likelihood.npz'
likelihood_viewer_path = project_dir / 'gpu_viewers' / 'likelihood_gpu_viewer.html'

In [ ]:
likelihood_result = create_likelihood_map(
    optimized_aligned_grid_path,
    floor_plan=likelihood_floor_plan,
    floor_cross=likelihood_floor_cross,
    length_unit=length_unit,
    chunk_voxels=likelihood_chunk_voxels,
    output_path=likelihood_map_path,
)

likelihood_map = likelihood_result.likelihood
valid_likelihood_overlap = likelihood_result.valid_overlap_mask
meta_likelihood = likelihood_result.metadata
x_likelihood, y_likelihood, z_likelihood = (
    meta_likelihood['coordinates'][axis] for axis in 'xyz'
)

print_likelihood_summary(likelihood_result, threshold=likelihood_threshold)

We make a histogram counting the number of voxels in the full volume according to their correlative likelihood score.

In [ ]:
likelihood_histogram_bins = 100
likelihood_histogram_log_y = True

likelihood_histogram_figure = make_likelihood_histogram(
    likelihood_result,
    threshold=likelihood_threshold,
    bins=likelihood_histogram_bins,
    colormap=likelihood_colormap,
    background_color=background_color,
    log_y=likelihood_histogram_log_y,
)
likelihood_histogram_figure

In [ ]:
likelihood_viewer_output = write_gpu_volume_viewer(
    likelihood_map,
    meta_likelihood,
    output_path=likelihood_viewer_path,
    threshold=likelihood_threshold,
    color='white',  # Used only if likelihood_colormap is None.
    colormap=likelihood_colormap,
    minimum_voxel_alpha=minimum_voxel_alpha,
    background_color=background_color,
    max_voxels=likelihood_max_display_voxels,
    voxel_scale=voxel_scale,
    open_browser=True,
)
likelihood_viewer_output

## Subvoxel atomic-center localization

Candidate centers are detected from the correlative likelihood score map with three-dimensional local maxima and a minimum physical separation. Each retained center is then refined against both aligned intensity volumes using one shared physical position with dataset-specific anisotropic Gaussian profiles. The result therefore depends on the saved alignment transform, likelihood floors, score threshold, minimum separation, and refinement controls. Colormap, sphere radius, marker size, opacity, and background color are display-only. The saved table contains physical `(x, y, z)` centers and fit diagnostics. The final viewer can use CPU-rendered Canvas 2D markers without WebGL or GPU-instanced spheres; neither mode imposes a display cap.

In [ ]:
# Localization uses the current aligned grid and correlative likelihood score map.
atom_score_threshold = likelihood_threshold
minimum_atom_separation = lat_param * np.sqrt(3) / 2 * 0.8 #80% of the distance between Sr and Ti atoms
atom_detection_smoothing_sigma = 0.0  # Physical units; 0 keeps the score map unsmoothed.
atom_fit_radius = minimum_atom_separation * 0.45
atom_maximum_subvoxel_shift = None  # None uses one aligned-grid sample.
atom_initial_step = None  # None uses one quarter of the finest grid spacing.
atom_position_tolerance = None  # None uses one thirty-second of that spacing.
atom_minimum_valid_fraction = 0.70

atom_output_prefix = project_dir / 'localized_atoms' / 'cmep_localized_atoms'
atom_cpu_marker_viewer_path = project_dir / 'cpu_viewers' / 'localized_atoms_marker_viewer.html'
atom_gpu_marker_viewer_path = project_dir / 'gpu_viewers' / 'localized_atoms_gpu_marker_viewer.html'
atom_gpu_sphere_viewer_path = project_dir / 'gpu_viewers' / 'localized_atoms_gpu_sphere_viewer.html'
atom_viewer_score_threshold = atom_score_threshold
atom_minimum_marker_alpha = minimum_voxel_alpha
atom_minimum_sphere_alpha = minimum_voxel_alpha
atom_initial_view_padding = 0.08
atom_initial_zoom_factor = 0.78  # Less than 1 starts farther out.
atom_background_color = background_color

In [ ]:
atom_localization_result = localize_atoms(
    likelihood_result,
    optimized_aligned_grid_path,
    lat_param=lat_param,
    score_threshold=atom_score_threshold,
    minimum_atom_separation=minimum_atom_separation,
    detection_smoothing_sigma=atom_detection_smoothing_sigma,
    fit_radius=atom_fit_radius,
    maximum_subvoxel_shift=atom_maximum_subvoxel_shift,
    initial_step=atom_initial_step,
    position_tolerance=atom_position_tolerance,
    minimum_valid_fraction=atom_minimum_valid_fraction,
    output_prefix=atom_output_prefix,
    progress=True,
)

atomic_positions_xyz = atom_localization_result.positions_xyz
atom_likelihood_scores = atom_localization_result.scores
meta_atoms = atom_localization_result.metadata
print_atom_localization_summary(atom_localization_result)

Visualize the 3D result

In [ ]:
atom_colormap = likelihood_colormap
atom_renderer = 'gpu'  # 'cpu' or 'gpu'.
atom_render_mode = 'markers'  # 'markers' or 'spheres'; spheres require GPU.

# Atom size parameters
atom_marker_size = 5.0  # Marker diameter in screen pixels; display only.
atom_sphere_radius = minimum_atom_separation * 0.20  # Display only.

atom_renderer_normalized = atom_renderer.strip().lower()
atom_render_mode_normalized = atom_render_mode.strip().lower()
if atom_renderer_normalized not in {'cpu', 'gpu'}:
    raise ValueError("atom_renderer must be 'cpu' or 'gpu'.")
if atom_render_mode_normalized not in {'markers', 'spheres'}:
    raise ValueError("atom_render_mode must be 'markers' or 'spheres'.")
if atom_renderer_normalized == 'cpu' and atom_render_mode_normalized == 'spheres':
    raise ValueError("Spheres require atom_renderer='gpu'; the CPU renderer supports markers.")

if atom_renderer_normalized == 'cpu':
    atom_viewer_output = write_cpu_atom_marker_viewer(
        atom_localization_result,
        output_path=atom_cpu_marker_viewer_path,
        marker_size=atom_marker_size,
        score_threshold=atom_viewer_score_threshold,
        color='white',  # Used only if atom_colormap is None.
        colormap=atom_colormap,
        minimum_marker_alpha=atom_minimum_marker_alpha,
        background_color=atom_background_color,
        initial_view_padding=atom_initial_view_padding,
        initial_zoom_factor=atom_initial_zoom_factor,
        open_browser=True,
    )
elif atom_render_mode_normalized == 'markers':
    atom_viewer_output = write_gpu_atom_marker_viewer(
        atom_localization_result,
        output_path=atom_gpu_marker_viewer_path,
        marker_size=atom_marker_size,
        score_threshold=atom_viewer_score_threshold,
        color='white',  # Used only if atom_colormap is None.
        colormap=atom_colormap,
        minimum_marker_alpha=atom_minimum_marker_alpha,
        background_color=atom_background_color,
        initial_view_padding=atom_initial_view_padding,
        initial_zoom_factor=atom_initial_zoom_factor,
        open_browser=True,
    )
else:
    atom_viewer_output = write_gpu_atom_viewer(
        atom_localization_result,
        output_path=atom_gpu_sphere_viewer_path,
        sphere_radius=atom_sphere_radius,
        score_threshold=atom_viewer_score_threshold,
        color='white',  # Used only if atom_colormap is None.
        colormap=atom_colormap,
        minimum_sphere_alpha=atom_minimum_sphere_alpha,
        background_color=atom_background_color,
        open_browser=True,
    )

atom_viewer_output